# Chapter 7 — Document Splitting

Hands-On LangChain. Code targets **LangChain 1.x** (splitters live in `langchain_text_splitters`).

After loading documents, you usually need to split them into smaller chunks — to fit a model's context
window and, more importantly, to keep *semantically related* text together so retrieval works well.
This chapter covers chunk size and overlap, the recursive and character splitters, splitting by tokens,
and structure-aware splitting on Markdown headers.

Companion notebook for Chapter 7.

```bash
pip install langchain-text-splitters tiktoken pypdf -q
```

## 7.1 Chunk size and overlap

Every splitter works with a **chunk size** (how big each chunk is) and a **chunk overlap** (how much
text is shared between consecutive chunks, like a sliding window — this keeps context from being lost
at a boundary). We use tiny values here so the behavior is easy to see.

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

chunk_size = 26
chunk_overlap = 4

r_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
c_splitter = CharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

A string exactly 26 characters long needs no splitting:

In [2]:
text1 = "abcdefghijklmnopqrstuvwxyz"
r_splitter.split_text(text1)

['abcdefghijklmnopqrstuvwxyz']

Make it longer than the chunk size and it splits, with the last 4 characters overlapping into the next chunk:

In [3]:
text2 = "abcdefghijklmnopqrstuvwxyzabcdefg"
r_splitter.split_text(text2)

['abcdefghijklmnopqrstuvwxyz', 'wxyzabcdefg']

With spaces, the text takes more room, so it splits into three chunks (the overlap counts spaces too):

In [4]:
text3 = "a b c d e f g h i j k l m n o p q r s t u v w x y z"
r_splitter.split_text(text3)

['a b c d e f g h i j k l m', 'l m n o p q r s t u v w x', 'w x y z']

The plain `CharacterTextSplitter` splits on a *single* separator — a newline by default. With no
newlines in `text3`, it would not split at all. Setting the separator to a space makes it behave like
the recursive splitter here:

In [5]:
c_splitter = CharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, separator=" ")
c_splitter.split_text(text3)

['a b c d e f g h i j k l m', 'l m n o p q r s t u v w x', 'w x y z']

## 7.2 Diving deeper into recursive splitting

On real prose, the difference matters. The recursive splitter tries a **list of separators in order** —
first paragraph breaks (`\n\n`), then line breaks (`\n`), then spaces, then individual characters —
so it prefers to break at natural boundaries.

In [6]:
text = """As the world grapples with the challenges of climate change, \
renewable energy emerges as a beacon of hope. Solar and wind power, \
in particular, are transforming the energy landscape, offering sustainable \
alternatives to traditional fossil fuels. \n\n \
Governments and businesses globally are investing in clean energy \
initiatives to reduce carbon footprints and mitigate environmental impact. \
The shift towards renewables not only addresses environmental concerns \
but also fosters innovation, creating a brighter and more sustainable \
future for generations to come."""
len(text)

562

In [7]:
c_splitter = CharacterTextSplitter(chunk_size=450, chunk_overlap=0, separator=" ")
r_splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=0,
                                            separators=["\n\n", "\n", " ", ""])

The character splitter breaks mid-sentence on a space:

In [8]:
c_splitter.split_text(text)

['As the world grapples with the challenges of climate change, renewable energy emerges as a beacon of hope. Solar and wind power, in particular, are transforming the energy landscape, offering sustainable alternatives to traditional fossil fuels. \n\n Governments and businesses globally are investing in clean energy initiatives to reduce carbon footprints and mitigate environmental impact. The shift towards renewables not only addresses',
 'environmental concerns but also fosters innovation, creating a brighter and more sustainable future for generations to come.']

The recursive splitter breaks on the paragraph boundary instead — cleaner chunks:

In [9]:
r_splitter.split_text(text)

['As the world grapples with the challenges of climate change, renewable energy emerges as a beacon of hope. Solar and wind power, in particular, are transforming the energy landscape, offering sustainable alternatives to traditional fossil fuels.',
 'Governments and businesses globally are investing in clean energy initiatives to reduce carbon footprints and mitigate environmental impact. The shift towards renewables not only addresses environmental concerns but also fosters innovation, creating a brighter and more sustainable future for generations to come.']

To split at sentence boundaries, add a separator for the period. A regex look-behind (`(?<=\. )`)
keeps the period attached to the sentence it ends:

In [10]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150, chunk_overlap=0,
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
    is_separator_regex=True,
)
r_splitter.split_text(text)

['As the world grapples with the challenges of climate change, renewable energy emerges as a beacon of hope.',
 'Solar and wind power, in particular, are transforming the energy landscape, offering sustainable alternatives to traditional fossil fuels.',
 'Governments and businesses globally are investing in clean energy initiatives to reduce carbon footprints and mitigate environmental impact.',
 'The shift towards renewables not only addresses environmental concerns but also fosters innovation, creating a brighter and more sustainable future',
 'for generations to come.']

## 7.3 Splitting a loaded PDF

On real documents you split the loaded `Document` objects with `split_documents`, which preserves each
chunk's metadata.

In [11]:
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("../data/career-in-ai.pdf").load()

text_splitter = CharacterTextSplitter(
    separator="\n", chunk_size=1000, chunk_overlap=150, length_function=len
)
docs = text_splitter.split_documents(pages)
print("pages:", len(pages), "-> chunks:", len(docs))

pages: 2 -> chunks: 5


In [12]:
print(docs[0].page_content[:200])
print("\nmetadata:", docs[0].metadata)

Building a Career in AI - Field Notes
An original summary of widely-known guidance on starting a career in artificial intelligence, written for this
book's examples.
Who shaped this advice
Much of the

metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-06-08T13:16:59+00:00', 'source': '../data/career-in-ai.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


## 7.4 Splitting by tokens

LLM context windows are measured in **tokens**, not characters, so splitting by token count maps more
directly to what the model sees. A token is often a word-piece rather than a whole word.

In [13]:
from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(chunk_size=1, chunk_overlap=0)
token_splitter.split_text("foo bar bazzyfoo")

['foo', ' bar', ' b', 'az', 'zy', 'foo']

Each item above is a single token. Applied to documents, `TokenTextSplitter` works just like the others:

In [14]:
token_doc_splitter = TokenTextSplitter(chunk_size=10, chunk_overlap=0)
token_docs = token_doc_splitter.split_documents(pages)
print(token_docs[0])

page_content='Building a Career in AI - Field Notes
An' metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-06-08T13:16:59+00:00', 'source': '../data/career-in-ai.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


## 7.5 Context-aware splitting on Markdown headers

Many documents have structure — Markdown headers, for instance — that we can split on explicitly.
`MarkdownHeaderTextSplitter` splits on headers and **records the header hierarchy in each chunk's
metadata**, so a chunk "remembers" which section it came from.

In [15]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_document = """# Title\n\n \
## Chapter 1\n\n \
Hi this is Chapter 1\n\n Hi this is Section 1\n\n \
### Section\n\n \
Hi this is Section 2 \n\n \
## Chapter 2\n\n \
Hi this is Chapter 2"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits[0]

Document(metadata={'Header 1': 'Title', 'Header 2': 'Chapter 1'}, page_content='Hi this is Chapter 1  \nHi this is Section 1')

The header path is captured in the metadata. A chunk from a sub-subsection carries all three header levels:

In [16]:
md_header_splits[1]

Document(metadata={'Header 1': 'Title', 'Header 2': 'Chapter 1', 'Header 3': 'Section'}, page_content='Hi this is Section 2')

The same splitter works on any real Markdown — for example files exported from Notion — letting
you carry section context into retrieval. With chunks and good metadata in hand, the next chapter moves
them into a **vector store**.